# 03 — Feature Engineering Analysis
**Source:** `data/silver/` (validated clean data)

**Goal:** Understand existing features, engineer new ones, and assess their predictive value for the `prix` target variable.

Sections:
1. Load Data
2. Analyse Existing Features
3. Engineer New Features
4. Correlation & Feature Importance
5. Feature Quality Summary

## 1 · Load Data

In [ ]:
import glob
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

BASE_DIR    = os.path.dirname(os.getcwd())
SILVER_DIR  = os.path.join(BASE_DIR, "data", "silver")
files = sorted(glob.glob(os.path.join(SILVER_DIR, "avito_clean_*.csv")))
assert files, f"No silver files in {SILVER_DIR}"

df = pd.read_csv(files[-1])

# ── Remove known bad rows ─────────────────────────────────────────────────────
# Rows where ville is empty or a known scraper artefact
BAD_VILLES = {"COURS ET FORMATIONS", "Cours Et Formations", ""}
df = df[~df["ville"].isin(BAD_VILLES)].reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")
print(f"Columns: {list(df.columns)}")

## 2 · Analyse Existing Features

In [ ]:
# ── Numeric features: describe ────────────────────────────────────────────────
NUM_COLS = [c for c in ["prix", "surface_m2", "nb_chambres",
                          "nb_salles_bain", "prix_par_m2", "age_bien"] if c in df.columns]

print("Numeric feature statistics:\n")
print(df[NUM_COLS].describe().round(2).to_string())

In [ ]:
# ── Distribution plots for numeric features ───────────────────────────────────
fig, axes = plt.subplots(2, len(NUM_COLS), figsize=(4 * len(NUM_COLS), 7))

for i, col in enumerate(NUM_COLS):
    s = df[col].dropna()
    # Histogram
    axes[0, i].hist(s, bins=15, color="#4C72B0", edgecolor="white", linewidth=0.5)
    axes[0, i].set_title(col, fontweight="bold")
    axes[0, i].set_xlabel("")
    # Boxplot
    axes[1, i].boxplot(s, vert=True, patch_artist=True,
                       boxprops=dict(facecolor="#4C72B0", alpha=0.7))
    axes[1, i].set_xticks([])

axes[0, 0].set_ylabel("Count")
axes[1, 0].set_ylabel("Value")
plt.suptitle("Numeric Feature Distributions", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical features: value counts ───────────────────────────────────────
CAT_COLS = [c for c in ["ville", "region_label", "categorie_prix", "prix_type"] if c in df.columns]

fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(5 * len(CAT_COLS), 5))
if len(CAT_COLS) == 1:
    axes = [axes]

for ax, col in zip(axes, CAT_COLS):
    counts = df[col].value_counts()
    counts.plot(kind="barh", ax=ax, color="#DD8452")
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("Count")
    ax.set_ylabel("")
    sns.despine(ax=ax)

plt.suptitle("Categorical Feature Distributions", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Prix vs categorical features (boxplots) ───────────────────────────────────
fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(6 * len(CAT_COLS), 5))
if len(CAT_COLS) == 1:
    axes = [axes]

for ax, col in zip(axes, CAT_COLS):
    order = df.groupby(col)["prix"].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df, x=col, y="prix", order=order,
        ax=ax, palette="Blues_d", width=0.5, fliersize=3
    )
    ax.set_title(f"Prix by {col}", fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
    sns.despine(ax=ax)

plt.suptitle("Price Distribution by Category", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3 · Engineer New Features

In [ ]:
df_feat = df.copy()

# ── F1: log_prix — reduce right skew in target ────────────────────────────────
df_feat["log_prix"] = np.log1p(df_feat["prix"])

# ── F2: log_surface — reduce right skew in surface ────────────────────────────
if "surface_m2" in df_feat.columns:
    df_feat["log_surface"] = np.log1p(df_feat["surface_m2"])

# ── F3: total_rooms = nb_chambres + nb_salles_bain ────────────────────────────
if "nb_chambres" in df_feat.columns and "nb_salles_bain" in df_feat.columns:
    df_feat["total_rooms"] = df_feat["nb_chambres"].fillna(0) + df_feat["nb_salles_bain"].fillna(0)

# ── F4: is_meuble — from titre keyword ────────────────────────────────────────
if "titre" in df_feat.columns:
    meuble_keywords = ["meublé", "meuble", "meubl"]
    df_feat["is_meuble"] = df_feat["titre"].str.lower().str.contains(
        "|".join(meuble_keywords), na=False
    ).astype(int)

# ── F5: is_bureau — office/commercial listing ─────────────────────────────────
if "titre" in df_feat.columns:
    bureau_keywords = ["bureau", "local", "plateau", "commercial", "professionnel"]
    df_feat["is_bureau"] = df_feat["titre"].str.lower().str.contains(
        "|".join(bureau_keywords), na=False
    ).astype(int)

# ── F6: has_surface — surface_m2 available ────────────────────────────────────
if "surface_m2" in df_feat.columns:
    df_feat["has_surface"] = df_feat["surface_m2"].notna().astype(int)

# ── F7: prix_par_chambre ──────────────────────────────────────────────────────
if "nb_chambres" in df_feat.columns:
    df_feat["prix_par_chambre"] = np.where(
        df_feat["nb_chambres"].notna() & (df_feat["nb_chambres"] > 0),
        df_feat["prix"] / df_feat["nb_chambres"],
        np.nan
    )

NEW_FEATURES = ["log_prix", "log_surface", "total_rooms",
                "is_meuble", "is_bureau", "has_surface", "prix_par_chambre"]
NEW_FEATURES = [f for f in NEW_FEATURES if f in df_feat.columns]

print("New features created:")
for f in NEW_FEATURES:
    non_null = df_feat[f].notna().sum()
    print(f"  • {f:<22}: {non_null}/{len(df_feat)} non-null  |  sample: {df_feat[f].dropna().head(3).values}")

In [ ]:
# ── Before/after: prix skew comparison ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

df_feat["prix"].dropna().hist(bins=20, ax=axes[0], color="#e74c3c", edgecolor="white")
axes[0].set_title(f"prix  (skew = {df_feat['prix'].skew():.2f})", fontweight="bold")
axes[0].set_xlabel("DH")

df_feat["log_prix"].dropna().hist(bins=20, ax=axes[1], color="#2ecc71", edgecolor="white")
axes[1].set_title(f"log_prix  (skew = {df_feat['log_prix'].skew():.2f})", fontweight="bold")
axes[1].set_xlabel("log(DH + 1)")

for ax in axes:
    sns.despine(ax=ax)
plt.suptitle("Log Transform — Reducing Right Skew in Target", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 4 · Correlation & Feature Importance

In [ ]:
# ── Pearson correlation with log_prix ─────────────────────────────────────────
TARGET = "log_prix"
CORR_COLS = [c for c in [
    "surface_m2", "log_surface", "nb_chambres", "nb_salles_bain",
    "total_rooms", "prix_par_m2", "prix_par_chambre",
    "is_meuble", "is_bureau", "is_grande_ville", "has_surface"
] if c in df_feat.columns]

df_feat["is_grande_ville"] = df_feat["is_grande_ville"].astype(int)

corr_target = (
    df_feat[CORR_COLS + [TARGET]]
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
    .sort_values(key=abs, ascending=False)
)

print(f"Pearson correlation with '{TARGET}':\n")
for feat, val in corr_target.items():
    bar = "█" * int(abs(val) * 20)
    direction = "+" if val > 0 else "-"
    print(f"  {feat:<25}: {direction}{abs(val):.3f}  {bar}")

In [ ]:
# ── Full correlation heatmap ──────────────────────────────────────────────────
heat_cols = [c for c in [
    TARGET, "surface_m2", "nb_chambres", "nb_salles_bain",
    "total_rooms", "prix_par_m2", "is_meuble", "is_bureau", "is_grande_ville"
] if c in df_feat.columns]

corr_matrix = df_feat[heat_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8}
)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: top correlated feature vs log_prix ───────────────────────────────
top_feature = corr_target.abs().idxmax()

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=df_feat.dropna(subset=[top_feature, TARGET]),
    x=top_feature, y=TARGET,
    hue="ville" if "ville" in df_feat.columns else None,
    s=70, alpha=0.8, ax=ax
)
ax.set_title(f"{top_feature} vs {TARGET}  (r = {corr_target[top_feature]:.3f})",
             fontweight="bold")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# ── Mean prix by ville ────────────────────────────────────────────────────────
if "ville" in df_feat.columns:
    ville_stats = (
        df_feat.groupby("ville")["prix"]
        .agg(count="count", mean="mean", median="median")
        .sort_values("median", ascending=False)
        .round(0)
    )
    print("Price statistics by city:\n")
    print(ville_stats.to_string())

    fig, ax = plt.subplots(figsize=(9, 4))
    ville_stats["median"].plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
    ax.set_title("Median Price by City (DH/month)", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("DH")
    ax.tick_params(axis="x", rotation=35)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

## 5 · Feature Quality Summary

In [ ]:
ALL_FEATURES = CORR_COLS + ["ville", "region_label", "categorie_prix"]
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_feat.columns]

summary_rows = []
for feat in ALL_FEATURES:
    null_pct  = round(100 * df_feat[feat].isna().mean(), 1)
    dtype     = str(df_feat[feat].dtype)
    corr_val  = corr_target.get(feat, None)
    corr_str  = f"{corr_val:+.3f}" if corr_val is not None else "n/a (categorical)"
    usable    = null_pct <= 50
    summary_rows.append({
        "Feature"   : feat,
        "dtype"     : dtype,
        "null_pct"  : null_pct,
        "corr_target": corr_str,
        "Usable"    : "✅" if usable else "⚠️"
    })

summary_df = pd.DataFrame(summary_rows)
print("Feature Quality Summary:\n")
print(summary_df.to_string(index=False))

print("\n✅ Features with ≤ 50% nulls are marked Usable.")
print("⚠️  Features with > 50% nulls should not be used as-is in ML models.")